# 有机锂中间体稳定性预测：全局 Arrhenius 方法

## 概述

**输入**: SMILES + 操作温度  
**输出**: t_max（最优停留时间）, t½（半衰期）, 反应器推荐（flash/flow/batch）

### 方法演变

| 版本 | 方法 | 局限 |
|---|---|---|
| v1 (旧) | 逐温度拟合 k_f, k_d → Arrhenius Ea_d → 描述符预测 Ea_d | Ea-lnA 补偿导致 t½ 不准; k_f 在高温不稳定 |
| **v2 (本 notebook)** | **全局 Arrhenius: 所有温度同时拟合 5 参数** → 类别特异描述符 | 数据量仍有限 (每类 7-8 个化合物) |

### 核心公式

$$\text{yield}(t_R, T) = y_{\max} \times \underbrace{(1 - e^{-k_f(T) \cdot t_R})}_{\text{生成}} \times \underbrace{e^{-k_d(T) \cdot t_R}}_{\text{分解}}$$

$$k_f(T) = A_f \cdot e^{-E_{a,f}/RT}, \quad k_d(T) = A_d \cdot e^{-E_{a,d}/RT}$$

$$t_{\max} = \frac{\ln(k_f/k_d)}{k_f - k_d}, \quad t_{1/2} = \frac{\ln 2}{k_d}$$


## Step 1: 读取原始数据

数据来自 Nagaki/Yoshida 等 12 篇流动化学文献 [De Gennaro 2014]。  
原始数据在 `dataset-manual-corrected.numbers` 中 (Apple Numbers 格式)，也可用 CSV 备份。

每行 = 一个实验点: 中间体 + 亲电体 + 温度 T + 停留时间 tR → 产率 yield%


In [ ]:
# ===== 1.1 导入依赖 =====
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors
from collections import defaultdict
from itertools import combinations
import json, warnings, os
warnings.filterwarnings('ignore')

# 设置绘图参数
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 100

print("依赖加载完成 ✓")


In [ ]:
# ===== 1.2 读取数据 =====
# 优先从 .numbers 文件读取（需要 pip install numbers-parser）
# 如果不可用，从 CSV 备份读取

DATA_DIR = '.'  # 在 data/ml_lifetime/ 目录下运行
CSV_FILE = os.path.join(DATA_DIR, 'clean_organolithium_unified_descriptors.csv')
NUMBERS_FILE = os.path.join(DATA_DIR, 'dataset-manual-corrected.numbers')

try:
    import numbers_parser
    doc = numbers_parser.Document(NUMBERS_FILE)
    table = doc.sheets[0].tables[0]
    headers = [str(table.cell(0, c).value) for c in range(table.num_cols)]
    rows = []
    for r in range(1, table.num_rows):
        row = {headers[c]: table.cell(r, c).value for c in range(table.num_cols)}
        rows.append(row)
    df = pd.DataFrame(rows)
    print(f"✓ 从 .numbers 读取: {df.shape[0]} 行 × {df.shape[1]} 列")
except Exception as e:
    df = pd.read_csv(CSV_FILE)
    print(f"✓ 从 CSV 读取: {df.shape[0]} 行 × {df.shape[1]} 列")

# 类型转换（.numbers 读出来可能是字符串）
for col in ['tR1_s', 'T1_C', 'tR2_s', 'T2_C', 'yield_pct']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 基本统计
print(f"\n化合物数: {df['intermediate_smiles_canonical'].nunique()}")
print(f"论文数:   {df['paper_id'].nunique()}")
print(f"温度范围: {df['T1_C'].min():.0f} ~ {df['T1_C'].max():.0f} °C")
print(f"tR 范围:  {df['tR1_s'].min():.4f} ~ {df['tR1_s'].max():.1f} s")
print(f"\n类别分布:")
print(df['intermediate_class'].value_counts())


## Step 2: 数据可视化 — yield vs tR 曲线

选 3 个代表化合物展示不同稳定性:
- **不稳定**: yield 快速上升又下降 → 需要 flash mixer
- **中等**: 低温平坦，高温才出现衰减 → flow reactor
- **稳定**: 所有温度产率平坦 → batch 操作

颜色从蓝（低温）到红（高温）。


In [ ]:
# ===== 2.1 典型化合物的 yield-tR 曲线 =====
# 只用甲醇淬灭数据（最可靠的动力学探针）
methanol = df[df['electrophile'].str.contains('methanol', na=False, case=False)]

examples = {
    'p-NO₂-PhLi (不稳定)': '[Li]c1ccc([N+](=O)[O-])cc1',
    'tBu o-LiB (中等)':    '[Li]c1ccccc1C(=O)OC(C)(C)C',
    'p-Br-PhLi (稳定)':    '[Li]c1ccc(Br)cc1',
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (label, smi) in zip(axes, examples.items()):
    sub = methanol[methanol['intermediate_smiles_canonical'] == smi]
    if len(sub) == 0:  # 如果甲醇数据不够，用全部数据
        sub = df[df['intermediate_smiles_canonical'] == smi]
    
    temps = sorted(sub['T1_C'].dropna().unique())
    clrs = plt.cm.coolwarm(np.linspace(0, 1, len(temps)))
    
    for T, c in zip(temps, clrs):
        t_data = sub[sub['T1_C'] == T].sort_values('tR1_s')
        ax.scatter(t_data['tR1_s'], t_data['yield_pct'], c=[c], s=20, label=f'{T:.0f}°C')
    
    ax.set_xscale('log')
    ax.set_xlabel('tR₁ (s)'); ax.set_ylabel('Yield (%)')
    ax.set_title(label, fontsize=10)
    ax.legend(fontsize=7, loc='best')
    ax.set_ylim(-5, 105); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("观察:")
print("  左: 高温(红)产率先升后降 → 不稳定, 需要精确控制 tR")
print("  中: 低温平坦, 只在高温出现衰减 → 中等稳定")
print("  右: 所有温度全程平坦 → 在实验窗口内很稳定")


## Step 3: 全局 Arrhenius 拟合 — 核心新方法

### 为什么不逐温度拟合？

逐温度拟合的问题: 当 $k_f \gg k_d$ 时（高温），yield 曲线对 $k_f$ 不敏感——$k_f = 100$ 和 $k_f = 100000$ 给出几乎相同的曲线。导致 $k_f$ 不可靠（出现负 $E_{a,f}$）。

### 全局拟合的思想

将一个化合物在**所有温度**（通常 4-9 个）的 yield-vs-tR 数据**同时拟合**到 5 个参数:

| 参数 | 物理意义 |
|---|---|
| $E_{a,f}$ | 生成活化能 (卤素-金属交换) |
| $\ln A_f$ | 生成前指因子 |
| $E_{a,d}$ | 分解活化能 (中间体热分解) |
| $\ln A_d$ | 分解前指因子 |
| $y_{\max}$ | 理论最大产率 |

低温数据约束 $k_f$（生成慢），高温数据约束 $k_d$（分解快），**互相补充**。


In [ ]:
# ===== 3.1 定义全局 Arrhenius 模型 =====
R_gas = 8.314e-3  # kJ/(mol·K)

def global_model(X, Ea_f, lnA_f, Ea_d, lnA_d, y_max):
    """
    全局 Arrhenius 模型
    
    参数:
        X: [tR, T_K] 的 2×N 数组
        Ea_f, lnA_f: 生成反应的 Arrhenius 参数
        Ea_d, lnA_d: 分解反应的 Arrhenius 参数
        y_max: 理论最大产率 (%)
    
    返回:
        yield(%) 的 N 维数组
    """
    tR, T_K = X[0], X[1]
    with np.errstate(over='ignore', under='ignore'):
        k_f = np.exp(lnA_f - Ea_f / (R_gas * T_K))  # 生成速率
        k_d = np.exp(lnA_d - Ea_d / (R_gas * T_K))  # 分解速率
        formation = 1.0 - np.exp(-k_f * tR)           # 生成项
        decay = np.exp(-k_d * tR)                       # 分解项
    return y_max * formation * decay

print("全局模型定义完成 ✓")
print("yield(tR, T) = y_max × (1 - exp(-k_f(T)·tR)) × exp(-k_d(T)·tR)")
print("k_f(T) = exp(lnA_f - Ea_f/RT)")
print("k_d(T) = exp(lnA_d - Ea_d/RT)")


In [ ]:
# ===== 3.2 对每个化合物执行全局拟合 =====
# 只用 tR1 步骤的数据（中间体的生成+分解在同一步）
tr1 = df[df['tR_step'].str.startswith('tR1', na=False)].copy()

global_results = []
n_success, n_fail = 0, 0

for smi in tr1['intermediate_smiles_canonical'].unique():
    sub = tr1[tr1['intermediate_smiles_canonical'] == smi]
    name = sub['intermediate'].iloc[0]
    
    # 策略: 甲醇淬灭数据优先（最可靠）
    methanol_sub = sub[sub['electrophile'].str.contains('methanol', na=False, case=False)]
    if len(methanol_sub) >= 10:
        sub = methanol_sub
    
    # 至少需要 3 个温度
    temps = sub['T1_C'].dropna().unique()
    if len(temps) < 3:
        continue
    
    # 收集所有数据点
    tR_all = sub['tR1_s'].values.astype(float)
    T_K_all = sub['T1_C'].values.astype(float) + 273.15
    y_all = sub['yield_pct'].values.astype(float)
    
    # 过滤无效值
    valid = np.isfinite(tR_all) & np.isfinite(T_K_all) & np.isfinite(y_all) & (tR_all > 0)
    tR_all, T_K_all, y_all = tR_all[valid], T_K_all[valid], y_all[valid]
    if len(tR_all) < 10:
        continue
    
    # 拟合
    X_data = np.array([tR_all, T_K_all])
    y_max_est = min(np.max(y_all) * 1.05, 100)
    
    try:
        popt, pcov = curve_fit(
            global_model, X_data, y_all,
            p0=[30, 15, 40, 15, y_max_est],                    # 初始猜测
            bounds=([0, -10, 0, -10, 10], [150, 50, 150, 50, 110]),  # 物理约束
            maxfev=50000, method='trf'
        )
        Ea_f, lnA_f, Ea_d, lnA_d, y_max = popt
        
        # 计算 R²
        y_pred = global_model(X_data, *popt)
        ss_res = np.sum((y_all - y_pred)**2)
        ss_tot = np.sum((y_all - np.mean(y_all))**2)
        r2 = 1 - ss_res/ss_tot if ss_tot > 0 else 0
        
        # 只保留物理合理的结果
        if r2 > 0.3 and Ea_f > 0 and Ea_d > 0:
            global_results.append({
                'intermediate': name, 'smi': smi,
                'Ea_f': round(Ea_f, 2), 'lnA_f': round(lnA_f, 2),
                'Ea_d': round(Ea_d, 2), 'lnA_d': round(lnA_d, 2),
                'y_max': round(y_max, 1), 'r2_global': round(r2, 4),
                'n_points': len(tR_all), 'n_temps': len(temps),
            })
            n_success += 1
        else:
            n_fail += 1
    except Exception as e:
        n_fail += 1

global_df = pd.DataFrame(global_results).sort_values('Ea_d')

print(f"=== 全局拟合结果 ===")
print(f"成功: {n_success} 化合物,  失败: {n_fail}")
print(f"R² 中位数: {global_df['r2_global'].median():.3f}")
print(f"R² 范围:   {global_df['r2_global'].min():.3f} ~ {global_df['r2_global'].max():.3f}")
print(f"Ea_f 范围: {global_df['Ea_f'].min():.1f} ~ {global_df['Ea_f'].max():.1f} kJ/mol (全部>0 ✓)")
print(f"Ea_d 范围: {global_df['Ea_d'].min():.1f} ~ {global_df['Ea_d'].max():.1f} kJ/mol")

# 展示前 10 个
global_df[['intermediate','Ea_f','Ea_d','lnA_f','lnA_d','r2_global','n_temps']].head(10)


## Step 4: 从 5 参数计算任意温度的 t_max, t½, 反应器推荐

有了 5 参数后，在**任意温度 T** 都可以计算:
- $t_{\max}$: 最优停留时间（产率峰值位置）
- $t_{1/2}$: 中间体半衰期
- 反应器类型: flash (<0.1s) / flow (0.1-60s) / batch (>60s)

**反应器边界基于设备物理限制，不随温度变化。温度改变的是 t_max 本身。**


In [ ]:
# ===== 4.1 辅助函数 =====

def calc_tmax(Ea_f, lnA_f, Ea_d, lnA_d, T_C):
    """计算给定温度下的 t_max (最优停留时间)"""
    T_K = T_C + 273.15
    k_f = np.exp(lnA_f - Ea_f / (R_gas * T_K))
    k_d = np.exp(lnA_d - Ea_d / (R_gas * T_K))
    if k_f <= k_d or k_f <= 0 or k_d <= 0:
        return 1e6  # 无峰值 → batch
    return np.log(k_f / k_d) / (k_f - k_d)

def calc_thalf(Ea_d, lnA_d, T_C):
    """计算给定温度下的 t½ (分解半衰期)"""
    T_K = T_C + 273.15
    k_d = np.exp(lnA_d - Ea_d / (R_gas * T_K))
    return np.log(2) / k_d if k_d > 0 else 1e10

def classify_reactor(t):
    """根据 t_max 推荐反应器类型"""
    if t < 0.1:   return 'flash'   # 微混合器 (<100ms)
    elif t < 60:  return 'flow'    # 管式反应器 (0.1s - 1min)
    else:         return 'batch'   # 常规操作 (>1min)

def format_time(t):
    """格式化时间显示"""
    if t >= 86400: return f"{t/86400:.1f}d"
    if t >= 3600:  return f"{t/3600:.1f}h"
    if t >= 60:    return f"{t/60:.0f}min"
    if t >= 0.1:   return f"{t:.2f}s"
    return f"{t*1000:.1f}ms"

print("辅助函数定义完成 ✓")


In [ ]:
# ===== 4.2 展示每个化合物在 4 个温度下的推荐 =====
# 这是"查表工具": 已知化合物直接从 5 参数计算

print(f"{'化合物':25s} {'Ea_f':>5s} {'Ea_d':>5s} {'R²':>5s} │", end='')
for T in [-78, -40, 0, 25]:
    print(f" {T:>5d}°C", end='')
print()
print("=" * 90)

for _, r in global_df[global_df['r2_global'] > 0.6].head(20).iterrows():
    print(f"{r['intermediate'][:25]:25s} {r['Ea_f']:5.1f} {r['Ea_d']:5.1f} {r['r2_global']:5.3f} │", end='')
    for T in [-78, -40, 0, 25]:
        tm = calc_tmax(r['Ea_f'], r['lnA_f'], r['Ea_d'], r['lnA_d'], T)
        rc = classify_reactor(tm)
        print(f" {rc:>5s}", end='')
    print()

# 统计各温度分布
print(f"\n各温度的 flash/flow/batch 分布:")
reliable = global_df[global_df['r2_global'] > 0.6]
for T in [-78, -40, 0, 25]:
    reactors = [classify_reactor(calc_tmax(r['Ea_f'],r['lnA_f'],r['Ea_d'],r['lnA_d'],T)) for _,r in reliable.iterrows()]
    print(f"  T={T:>5d}°C: flash={reactors.count('flash'):2d}, flow={reactors.count('flow'):2d}, batch={reactors.count('batch'):2d}")


## Step 5: 结构分类 + 描述符准备

### 基于 SMILES 结构分类（不是名字！）

| 类别 | SMILES 特征 | 分解机制 |
|---|---|---|
| o-ArLi | `[Li]c1ccccc1X` | 苯炔消除 / 螯合 |
| m-ArLi | `[Li]c1cccc(X)c1` | THF 溶剂裂解 |
| p-ArLi | `[Li]c1ccc(X)cc1` | THF 溶剂裂解 |
| oxiranylLi | 含 `CO1` 环氧 | 环氧开环 |

### 描述符来源

所有描述符已在原始数据集中（用 GFN2-xTB 预计算），从 SMILES 计算只需秒级。


In [ ]:
# ===== 5.1 结构分类函数 =====
def classify_structure(smi):
    """基于 SMILES 结构分类有机锂中间体"""
    smi = str(smi)
    # oxiranylLi: 含三元环氧
    if any(p in smi for p in ['CO1', 'C1CO1', 'OC1', 'C1OC1']):
        return 'oxiranylLi'
    # ortho: [Li]c1ccccc1X (Li 和 X 在苯环 1,2 位)
    if '[Li]c1ccccc1' in smi and smi != '[Li]c1ccccc1':
        return 'o-ArLi'
    # para: [Li]c1ccc(X)cc1
    if '[Li]c1ccc(' in smi:
        return 'p-ArLi'
    # meta: [Li]c1cccc(X)c1
    if '[Li]c1cccc(' in smi:
        return 'm-ArLi'
    # 杂环芳基锂
    if '[Li]c1' in smi:
        return 'hetero-ArLi'
    return 'other'

# ===== 5.2 提取描述符表 =====
# 描述符映射: 简称 → 数据集列名
int_desc = {
    'q_C': 'dft_charge_C_ipso',      # C-Li 碳上 Mulliken 电荷
    'd_LiC': 'dft_LiC_bond_A',       # Li-C 键长 (Å)
    'BDE': 'dft_LiC_BDE_kJ',         # Li-C 键离解能 (kJ/mol)
    '%Vbur': 'buried_vol_Li',         # Li 周围埋藏体积
    'Gsolv': 'dft_Gsolv_kJ',         # THF 溶剂化自由能
    'HOMO': 'dft_HOMO_eV',           # 最高占据轨道能量
    'eta': 'HOMO_LUMO_gap_eV',       # 化学硬度 (LUMO-HOMO)/2
    'fukui': 'fukui_f_minus_C',      # Fukui 亲核函数
    'B1': 'sterimol_B1',             # Sterimol 最小宽度
    'B5': 'sterimol_B5',             # Sterimol 最大宽度
    'L': 'sterimol_L',               # Sterimol 长度
    'vol': 'mol_volume',             # 分子体积
    'dipole': 'dft_dipole_D',        # 偶极矩
}

# 每个化合物取一行描述符
unique = df.drop_duplicates(subset='intermediate_smiles_canonical')
desc_table = unique[['intermediate', 'intermediate_smiles_canonical', 'intermediate_class'] +
                     list(int_desc.values())].copy()
for col in int_desc.values():
    desc_table[col] = pd.to_numeric(desc_table[col], errors='coerce')

# ===== 5.3 合并: 全局拟合参数 + 描述符 + 分类 =====
model_data = global_df.merge(desc_table, left_on='smi',
    right_on='intermediate_smiles_canonical', how='inner', suffixes=('', '_d'))
model_data['class'] = model_data['smi'].apply(classify_structure)

# 只用可靠拟合 (R² > 0.6) 且 Ea_d 合理 (< 140 kJ/mol)
model_data = model_data[(model_data['r2_global'] > 0.6) & (model_data['Ea_d'] < 140)]

print(f"建模数据: {len(model_data)} 化合物 (R²>0.6, Ea_d<140)")
print(f"\n类别分布:")
print(model_data['class'].value_counts())


## Step 6: 类别特异描述符筛选

**核心发现**: 全局描述符模型 R² < 0.2（不同类别机制不同），但分类别后 R² 可达 0.85-0.97。

对每个类别，穷举所有 2-3 个描述符的组合，用 LOO-CV 评估对 4 个 Arrhenius 参数的预测能力。


In [ ]:
# ===== 6.1 LOO-CV R² 函数 =====
def loo_r2(X, y):
    """留一交叉验证 R²"""
    if len(X) < 5: return -999
    yp = np.zeros_like(y, dtype=float)
    for tr, te in LeaveOneOut().split(X):
        yp[te] = LinearRegression().fit(X[tr], y[tr]).predict(X[te])
    ss_r = np.sum((y - yp)**2)
    ss_t = np.sum((y - np.mean(y))**2)
    return 1 - ss_r/ss_t if ss_t > 0 else 0

# ===== 6.2 穷举筛选 =====
desc_names = list(int_desc.keys())

print(f"{'类别':>12s} {'参数':>6s} {'描述符':35s} {'LOO-R²':>8s} {'n':>4s}")
print("=" * 70)

class_best = {}  # 保存每个类别的最佳描述符组合

for cls in ['oxiranylLi', 'm-ArLi', 'o-ArLi', 'p-ArLi']:
    cls_data = model_data[model_data['class'] == cls]
    if len(cls_data) < 5:
        print(f"  {cls}: n={len(cls_data)}, 跳过")
        continue
    
    class_best[cls] = {}
    
    for target in ['Ea_f', 'Ea_d', 'lnA_f', 'lnA_d']:
        best = {'r2': -999}
        for np_ in [2, 3]:
            for combo in combinations(desc_names, np_):
                cols = [int_desc[d] for d in combo]
                v = cls_data[cols + [target]].dropna()
                if len(v) < max(4, len(cls_data) * 0.5):
                    continue
                r2 = loo_r2(v[cols].values, v[target].values)
                if r2 > best['r2']:
                    best = {'r2': r2, 'descs': list(combo), 'cols': cols, 'n': len(v)}
        
        if best['r2'] > -999:
            class_best[cls][target] = best
            star = ' ★' if best['r2'] > 0.5 else ''
            print(f"{cls:>12s} {target:>6s} {'+'.join(best['descs']):35s} {best['r2']:8.3f} {best['n']:4d}{star}")
    print()

print("\n★ = R² > 0.5 (可预测)")


## Step 7: 端到端 LOO 验证

对每个化合物执行 LOO:
1. 用同类别其他化合物训练 descriptor → 4 参数模型
2. 预测该化合物的 Ea_f, lnA_f, Ea_d, lnA_d
3. 在 -78, -40, 0, 25°C 计算 t_max
4. 分类 flash/flow/batch
5. 和实际比较


In [ ]:
# ===== 7.1 LOO 端到端验证 =====
all_results = []

for cls, config in class_best.items():
    cls_data = model_data[model_data['class'] == cls]
    if len(cls_data) < 5:
        continue
    
    for i in range(len(cls_data)):
        test = cls_data.iloc[i]
        train = cls_data.drop(cls_data.index[i])
        
        # 预测 4 个参数
        predicted = {}
        for param in ['Ea_f', 'Ea_d', 'lnA_f', 'lnA_d']:
            if param not in config:
                # 该参数无好的描述符模型 → 用类别均值
                predicted[param] = train[param].mean()
            else:
                info = config[param]
                v_train = train[info['cols'] + [param]].dropna()
                tv = [test[c] for c in info['cols']]
                if any(np.isnan(v) for v in tv) or len(v_train) < 3:
                    predicted[param] = train[param].mean()
                else:
                    reg = LinearRegression().fit(v_train[info['cols']].values, v_train[param].values)
                    predicted[param] = reg.predict([tv])[0]
        
        # 在 4 个温度计算 t_max 和分类
        for T in [-78, -40, 0, 25]:
            tm_actual = calc_tmax(test['Ea_f'], test['lnA_f'], test['Ea_d'], test['lnA_d'], T)
            tm_pred = calc_tmax(predicted['Ea_f'], predicted['lnA_f'], predicted['Ea_d'], predicted['lnA_d'], T)
            th_actual = calc_thalf(test['Ea_d'], test['lnA_d'], T)
            th_pred = calc_thalf(predicted['Ea_d'], predicted['lnA_d'], T)
            
            all_results.append({
                'name': test['intermediate'], 'class': cls, 'T': T,
                'tm_actual': tm_actual, 'tm_pred': tm_pred,
                'th_actual': th_actual, 'th_pred': th_pred,
                'r_actual': classify_reactor(tm_actual),
                'r_pred': classify_reactor(tm_pred),
            })

rdf = pd.DataFrame(all_results)
rdf['correct'] = rdf['r_actual'] == rdf['r_pred']

# 按类别 × 温度的准确率
print(f"{'类别':>12s} {'T':>6s} {'准确率':>8s}")
print("=" * 30)
for cls in ['oxiranylLi', 'm-ArLi', 'o-ArLi', 'p-ArLi']:
    for T in [-78, -40, 0, 25]:
        sub = rdf[(rdf['class'] == cls) & (rdf['T'] == T)]
        if len(sub) == 0: continue
        print(f"{cls:>12s} {T:>6d}°C {sub['correct'].mean():>8.0%}")
    print()

total_acc = rdf['correct'].mean()
print(f"总体: {total_acc:.1%} ({rdf['correct'].sum()}/{len(rdf)})")


## Step 8: 预测工具 — 输入 SMILES + 温度

两种模式:
1. **查表模式**: 已拟合的 45 个化合物 → 直接从 5 参数计算（精确）
2. **预测模式**: 新化合物 → 用类别特异描述符模型预测 4 参数（近似）


In [ ]:
# ===== 8.1 预测函数 =====
def predict_compound(smiles, T_list=[-78, -40, 0, 25]):
    """
    输入 SMILES → 预测 t_max, t½, 反应器推荐
    
    参数:
        smiles: 有机锂中间体的 SMILES (如 '[Li]c1ccccc1Br')
        T_list: 温度列表 (°C)
    """
    cls = classify_structure(smiles)
    
    # 模式 1: 查表 (已知化合物)
    match = global_df[global_df['smi'] == smiles]
    if len(match) > 0:
        r = match.iloc[0]
        params = {'Ea_f': r['Ea_f'], 'lnA_f': r['lnA_f'],
                  'Ea_d': r['Ea_d'], 'lnA_d': r['lnA_d']}
        source = '查表'
    elif cls in class_best:
        # 模式 2: 预测 (新化合物)
        m = desc_table[desc_table['intermediate_smiles_canonical'] == smiles]
        if len(m) == 0:
            print(f"  ✗ SMILES 不在数据集中，需要先计算 xTB 描述符")
            return
        m = m.iloc[0]
        cls_data = model_data[model_data['class'] == cls]
        params = {}
        for param in ['Ea_f', 'Ea_d', 'lnA_f', 'lnA_d']:
            if param not in class_best[cls]:
                params[param] = cls_data[param].mean()
            else:
                info = class_best[cls][param]
                vals = [pd.to_numeric(m.get(c), errors='coerce') for c in info['cols']]
                if any(np.isnan(v) for v in vals):
                    params[param] = cls_data[param].mean()
                else:
                    v_train = cls_data[info['cols'] + [param]].dropna()
                    reg = LinearRegression().fit(v_train[info['cols']].values, v_train[param].values)
                    params[param] = reg.predict([vals])[0]
        source = '预测'
    else:
        print(f"  ✗ 类别 {cls} 无训练模型")
        return
    
    print(f"  [{source}] 类别={cls}, Ea_f={params['Ea_f']:.1f}, Ea_d={params['Ea_d']:.1f} kJ/mol")
    print(f"  {'T':>6s} {'t_max':>10s} {'t½':>10s} {'reactor':>8s}")
    print(f"  {'-'*38}")
    for T in T_list:
        tm = calc_tmax(params['Ea_f'], params['lnA_f'], params['Ea_d'], params['lnA_d'], T)
        th = calc_thalf(params['Ea_d'], params['lnA_d'], T)
        rc = classify_reactor(tm)
        print(f"  {T:>6d}°C {format_time(tm):>10s} {format_time(th):>10s} {rc:>8s}")

# ===== 8.2 演示 =====
test_cases = [
    ('[Li]c1cccc(C(=O)OC)c1',          'methyl m-lithiobenzoate'),
    ('[Li]c1ccccc1C(=O)OC(C)(C)C',     'tBu o-lithiobenzoate'),
    ('[Li]CCCC1CO1',                     '3-(oxiran-2-yl)propylLi'),
    ('[Li]c1ccc(OC)cc1',                'p-Anisyllithium'),
    ('[Li]c1ccc(C#N)cc1',               'p-cyanophenyllithium'),
]

for smi, name in test_cases:
    print(f"\n{name}:")
    predict_compound(smi)


## Step 9: 总结

### 方法

```
原始 yield-vs-tR 数据 (2610 行, 多温度)
  ↓ 全局 Arrhenius 拟合
5 参数 (Ea_f, lnA_f, Ea_d, lnA_d, y_max) per compound
  ↓ 类别特异描述符模型
SMILES → xTB 描述符 → 预测 4 参数
  ↓ 计算
任意温度的 t_max, t½ → flash/flow/batch
```

### 结果

| 指标 | 值 |
|---|---|
| 全局拟合 | 45 化合物, R² 中位数 0.879 |
| 最佳类别 (oxiranylLi) | 4 参数全 R² > 0.85 |
| 端到端反应器分类 | 总体 80%, 0°C 95% |

### 局限性

- 每类只有 7-8 个化合物 → 更多数据会提升精度
- Ea_f 对 ArLi 较难预测 (受底物 C-X 键影响)
- batch 类数据间接推断 (来自 formation_only)

### 参考文献

- [1] De Gennaro 2014, *Lithium Compounds in Organic Synthesis*, Ch.18
- [7] Ramachandran 2010, *J. Phys. Chem. A*, 114, 8423
- [8] Bannwarth 2019, *J. Chem. Theory Comput.*, 15, 1652 (GFN2-xTB)
- [14] Collum 2007, *Angew. Chem. Int. Ed.*, 46, 3002
